# Weekly project

Today you are going to implement the last parts of the algorithm you started on monday. For reference you can see it below.

![img](algorithm_3.png "Title")


It is a good idea to follow and track the steps in the algorithm in the below implementation. Only take one step at a time.

Once you have the algorithm up and running you can try with a larger dataset to see if your algorithm is able to maintain good accurracy over a longer distance. The larger dataset can be found here:
[Left images](https://dtudk-my.sharepoint.com/:u:/g/personal/evanb_dtu_dk/EQu8kmGBDDROtGJ7IkZB2tQBJrxmgY9t8LVM_JuEi83TYw)
[Right images](https://dtudk-my.sharepoint.com/:u:/g/personal/evanb_dtu_dk/EcKI_zrXTvpMulizidCZm4oBLJcQ_LTV9Zs6oQFF74JTRQ)

In [1]:
import numpy as np
import cv2 as cv2
import glob

def getK():
    return np.array([[7.188560e+02, 0.000000e+00, 6.071928e+02],
                     [0, 7.188560e+02, 1.852157e+02],
                     [0, 0, 1]])

def getTruePose():
    file = '00.txt'
    return np.genfromtxt(file, delimiter=' ', dtype=None)

<details>
  <summary>tips:</summary>

- [Feature Matching](https://docs.opencv.org/4.x/dc/dc3/tutorial_py_matcher.html)
- To get the keypoint of a certain match do: ```kp1[match.queryIdx].pt``` and ```kp2[match.trainIdx].pt```

In [2]:
def extract_keypoints_sift(img1, img2, K, baseline):
    """
    Use SIFT to detect keypoints and compute descriptors in both images,
    and find matches between them using knnMatch with k=2
    """
    sift = cv2.SIFT_create()
    kp1, desc1 = sift.detectAndCompute(img1, None)
    kp2, desc2 = sift.detectAndCompute(img2, None)
    """
    Get the best keypoints by applying the Lowes ratio test
    """
    bf = cv2.BFMatcher()
    raw_matches = bf.knnMatch(desc1, desc2, k=2)

    good_matches = []
    for m, n in raw_matches:
        if m.distance < 0.75 * n.distance:
            good_matches.append(m)

    # Transform best keypoints to nparray
    p1 = np.array([kp1[m.queryIdx].pt for m in good_matches]).astype('float32')
    p2 = np.array([kp2[m.trainIdx].pt for m in good_matches]).astype('float32')

    ##### ############# ##########
    ##### Do Triangulation #######
    ##### ########################
    # project the feature points to 3D with triangulation
    # projection matrix for Left and Right Image
    M_left = K.dot(np.hstack((np.eye(3), np.zeros((3, 1)))))
    M_rght = K.dot(np.hstack((np.eye(3), np.array([[-baseline, 0, 0]]).T)))

    p1_flip = np.vstack((p1.T, np.ones((1, p1.shape[0]))))
    p2_flip = np.vstack((p2.T, np.ones((1, p2.shape[0]))))

    P = cv2.triangulatePoints(M_left, M_rght, p1_flip[:2], p2_flip[:2])

    # Normalize homogeneous coordinates (P->Nx4  [N,4] is the normalizer/scale)
    P = P / P[3]
    land_points = P[:3]

    return land_points.T, p1

In [3]:
def featureTracking(prev_img, next_img, prev_points, world_points):
    """
    Use OpenCV to find the prev_points from the prev_img in the next_img
    Remember to remove points that could not be found from prev_points, next_points, and world_points
    hint: status == 1
    """
    params = dict(winSize=(21, 21),
                 maxLevel=3,
                 criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 30, 0.01))

    prev_points = np.asarray(prev_points, dtype=np.float32).reshape(-1, 1, 2)

    next_points, status, _ = cv2.calcOpticalFlowPyrLK(
        prev_img, next_img, prev_points, None, **params
    )

    status = status.reshape(-1).astype(bool)
    world_points = world_points[status]
    prev_points = prev_points[status]
    next_points = next_points[status]

    return world_points, prev_points, next_points


In [ ]:
def playImageSequence(l_img_paths, r_img_paths, K):
    baseline = 0.54
    left_img = cv2.imread(l_img_paths[0], 0)
    right_img = cv2.imread(r_img_paths[0], 0)

    ##### ################################# #######
    ##### Get 3D points Using Triangulation #######
    ##### #########################################
    """
    Implement step 1.2 and 1.3
    Store the features in 'reference_2D' and the 3D points (landmarks) in 'landmark_3D'
    hint: use 'extract_keypoints_sift' above
    """
    landmark_3D, reference_2D = extract_keypoints_sift(left_img, right_img, K=K, baseline=baseline)


    # reference
    reference_img = left_img

    # Groundtruth for plot
    truePose = getTruePose()
    traj = np.zeros((600, 600, 3), dtype=np.uint8)
    maxError = 0

    for i in range(1, len(l_img_paths)):
        #Make sure not to print every line
        toPrint = i%15==0
        if toPrint:
            print('Image: ', i)
        curImage = cv2.imread(l_img_paths[i], 0)
        curImage_R = cv2.imread(r_img_paths[i], 0)

        ##### ############################################################# #######
        ##### Calculate 2D and 3D feature correspndances in t=T-1, and t=T  #######
        ##### #####################################################################
        """
        Implement step 2.2)
        Remember this is a part of a loop, so the initial features are already
        provided in step 1)-1.3) outside the loop in 'reference_2D' and 'landmark_3D'
        """

        landmark_3D, reference_2D, track_2D =  featureTracking(reference_img, curImage, reference_2D, landmark_3D)
    
        ##### ##############################################################
        ##### Calculate relative pose using PNPRansac in world frame #######
        ##### ##############################################################
        """
        Implement step 2.3)
        """
        _, rvec, tvec, inliers = cv2.solvePnPRansac(landmark_3D, track_2D, K, None)
        if inliers is not None:
            inliers = inliers.flatten()
            landmark_3D = landmark_3D[inliers]
            reference_2D = reference_2D[inliers]

        ##### ####################################################### #######
        ##### Get Pose and Tranformation Matrix in world coordionates #######
        ##### ###############################################################
        rot, _ = cv2.Rodrigues(rvec)
        tvec = -rot.T.dot(tvec)  # coordinate transformation, from camera to world. What is the XYZ of the camera wrt World
        inv_transform = np.hstack((rot.T, tvec))  # inverse transform. A tranform projecting points from the camera frame to the world frame

        ##### ################################# #######
        ##### Get 3D points Using Triangulation #######
        ##### #########################################
        # re-obtain the 3D points. 
        # hint: use 'extract_keypoints_sift' again
        """
        Implement step 2.4)
        """

        landmark_3D_new, reference_2D_new = extract_keypoints_sift(curImage, curImage_R, K=K, baseline=baseline)
        
        #Project the points from camera to world coordinates
        reference_2D = reference_2D_new.astype('float32')
        landmark_3D = inv_transform.dot(np.vstack((landmark_3D_new.T, np.ones((1, landmark_3D_new.shape[0])))))
        landmark_3D = landmark_3D.T

        ##### ####################### #######
        ##### Done, Next image please #######
        ##### ###############################
        reference_img = curImage

        ##### ################################## #######
        ##### START OF Print and visualize stuff #######
        ##### ##########################################
        # draw images

        x, y, z = tvec.ravel()

        draw_x, draw_y = int(x) + 300, 600 - (int(z) + 100)
        true_x, true_y = int(truePose[i][3]) + 300, 600-(int(truePose[i][11]) + 100)

        curError = np.sqrt(
            (tvec[0] - truePose[i][3]) ** 2 +
            (tvec[1] - truePose[i][7]) ** 2 +
            (tvec[2] - truePose[i][11]) ** 2)
        
        if (curError > maxError):
            maxError = curError
        if toPrint:
            print(f"Estimated position, X : {tvec[0]} , Y : {tvec[1]} , Z : {tvec[2]}")
            print(f"True position, X : {truePose[i][3]} , Y : {truePose[i][7]} , Z : {truePose[i][11]}")
            print("-")
            print("Rotation vector","[{0:02f}, {1:02f}, {2:02f}]".format(rvec[0][0], rvec[1][0], rvec[2][0]))
            print("\n-----\n")

        text = "Coordinates: x ={0:02f}m y = {1:02f}m z = {2:02f}m".format(x, y, z)
        
        cv2.circle(traj, (draw_x, draw_y), 1, (0, 0, 255), 2)
        cv2.circle(traj, (true_x, true_y), 1, (255, 0, 0), 2)
        cv2.rectangle(traj, (10, 30), (550, 50), (0, 0, 0), cv2.FILLED)
        cv2.putText(traj, text, (10, 50), cv2.FONT_HERSHEY_PLAIN, 1, (255, 255, 255), 1, 8)


    

        h1, w1 = traj.shape[:2]
        h2, w2 = curImage.shape[:2]
        vis = np.zeros((max(h1, h2), w1 + w2, 3), np.uint8)
        vis[:h1, :w1, :3] = traj

        curImage_color = np.dstack((curImage, curImage, curImage)).copy()
        for pt in track_2D.reshape(-1, 2):
            cv2.circle(curImage_color, (int(pt[0]), int(pt[1])), 3, (0, 255, 0), -1)
        vis[:h2, w1:w1 + w2, :3] = curImage_color

        cv2.imshow("Trajectory", vis)
        k = cv2.waitKey(1) & 0xFF
        if k == 27: break


    cv2.waitKey(0)
    cv2.destroyAllWindows()
    print('Maximum Error: ', maxError)
    ##### ################################ #######
    ##### END OF Print and visualize stuff #######
    ##### ########################################

In [5]:
# Load image paths
left_img_paths = sorted(glob.glob('left/*.png'))
right_img_paths = sorted(glob.glob('right/*.png'))

K = getK()

playImageSequence(left_img_paths, right_img_paths, K)

QFontDatabase: Cannot find font directory /home/s230642/.local/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/s230642/.local/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/s230642/.local/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/s230642/.local/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/s230642/.local/lib/python3.10/si

Image:  15
Estimated position, X : [-0.44710271] , Y : [-0.14194546] , Z : [11.68097009]
True position, X : -0.7018788 , Y : -0.4239119 , Z : 12.86965
-
Rotation vector [-0.002759, 0.053200, 0.001114]

-----

Maximum Error:  [1.24794866]


# Challenge 
The current implementation only uses features computed at the current timestep. However, as we process more images we potentially have a lot of features from previous timesteps that are still valid. The challenge is to expand the `extract_keypoints_surf(..., refPoints)` function by giving it old reference points. You should then combine your freshly computed features with the old features and remove all duplicates. This requires you to keep track of old features and 3D points.

Hint 1: The following function `removeDuplicate` can be used for removing duplicates.

Hint 2: you are not interested in points that are behind you, so remember to remove points that are negative in the direction you move.

In [9]:
def removeDuplicate(queryPoints, refPoints, radius=5):
    # remove duplicate points from new query points,
    for i in range(len(queryPoints)):
        query = queryPoints[i]
        xliml, xlimh = query[0] - radius, query[0] + radius
        yliml, ylimh = query[1] - radius, query[1] + radius
        inside_x_lim_mask = (refPoints[:, 0] > xliml) & (refPoints[:, 0] < xlimh)
        curr_kps_in_x_lim = refPoints[inside_x_lim_mask]

        if curr_kps_in_x_lim.shape[0] != 0:
            inside_y_lim_mask = (curr_kps_in_x_lim[:, 1] > yliml) & (curr_kps_in_x_lim[:, 1] < ylimh)
            curr_kps_in_x_lim_and_y_lim = curr_kps_in_x_lim[inside_y_lim_mask, :]
            if curr_kps_in_x_lim_and_y_lim.shape[0] != 0:
                queryPoints[i] = np.array([0, 0])
    return (queryPoints[:, 0] != 0)

The code below implements the challange by extending the algorimth to have points carry over from previous frames. In order do this it merged new points with previus points, and then drops them once they go behind the camera

In [10]:
def playImageSequenceChallange(l_img_paths, r_img_paths, K):
    baseline = 0.54
    left_img = cv2.imread(l_img_paths[0], 0)
    right_img = cv2.imread(r_img_paths[0], 0)

    ##### ################################# #######
    ##### Get 3D points Using Triangulation #######
    ##### #########################################
    """
    Implement step 1.2 and 1.3
    Store the features in 'reference_2D' and the 3D points (landmarks) in 'landmark_3D'
    hint: use 'extract_keypoints_sift' above
    """
    landmark_3D, reference_2D = extract_keypoints_sift(left_img, right_img, K=K, baseline=baseline)


    # reference
    reference_img = left_img

    # Groundtruth for plot
    truePose = getTruePose()
    traj = np.zeros((600, 600, 3), dtype=np.uint8)
    maxError = 0

    for i in range(1, len(l_img_paths)):
        
        #Make sure not to print every line
        toPrint = i%15==0
        if toPrint:
            print('Image: ', i)
        curImage = cv2.imread(l_img_paths[i], 0)
        curImage_R = cv2.imread(r_img_paths[i], 0)

        ##### ############################################################# #######
        ##### Calculate 2D and 3D feature correspndances in t=T-1, and t=T  #######
        ##### #####################################################################
        """
        Implement step 2.2)
        Remember this is a part of a loop, so the initial features are already
        provided in step 1)-1.3) outside the loop in 'reference_2D' and 'landmark_3D'
        """

        landmark_3D, reference_2D, track_2D =  featureTracking(reference_img, curImage, reference_2D, landmark_3D)
        track_2D = track_2D.reshape(-1, 2)  # calcOpticalFlowPyrLK returns (N,1,2); normalize to (N,2)

        ##### ##############################################################
        ##### Calculate relative pose using PNPRansac in world frame #######
        ##### ##############################################################
        """
        Implement step 2.3)
        """
        _, rvec, tvec, inliers = cv2.solvePnPRansac(landmark_3D, track_2D, K, None)
        if inliers is not None:
            inliers = inliers.flatten()
            landmark_3D = landmark_3D[inliers]
            reference_2D = reference_2D[inliers]
            track_2D = track_2D[inliers]

        ##### ####################################################### #######
        ##### Get Pose and Tranformation Matrix in world coordionates #######
        ##### ###############################################################
        rot, _ = cv2.Rodrigues(rvec)
        tvec = -rot.T.dot(tvec)  # coordinate transformation, from camera to world. What is the XYZ of the camera wrt World
        inv_transform = np.hstack((rot.T, tvec))  # inverse transform. A tranform projecting points from the camera frame to the world frame

        ##### ##################################################### #######
        ##### fresh SIFT features instead of discarding everything  #######
        ##### ##################################################### #######
        landmark_3D_new, reference_2D_new = extract_keypoints_sift(curImage, curImage_R, K=K, baseline=baseline)

        # drop candidates behind the camera (negative depth along viewing axis)
        valid_depth = landmark_3D_new[:, 2] > 0
        landmark_3D_new = landmark_3D_new[valid_depth]
        reference_2D_new = reference_2D_new[valid_depth]

        keep_mask = removeDuplicate(reference_2D_new, track_2D)
        landmark_3D_new = landmark_3D_new[keep_mask]
        reference_2D_new = reference_2D_new[keep_mask]

        # add new-points into the world frame and merge with the carried-over ones
        if landmark_3D_new.shape[0] > 0:
            landmark_3D_new_world = inv_transform.dot(
                np.vstack((landmark_3D_new.T, np.ones((1, landmark_3D_new.shape[0]))))
            ).T
            landmark_3D = np.vstack((landmark_3D, landmark_3D_new_world)).astype('float32')
            reference_2D = np.vstack((track_2D, reference_2D_new)).astype('float32')
        else:
            landmark_3D = landmark_3D.astype('float32')
            reference_2D = track_2D.astype('float32')

        ##### ####################### #######
        ##### Done, Next image please #######
        ##### ###############################
        reference_img = curImage

        ##### ################################## #######
        ##### START OF Print and visualize stuff #######
        ##### ##########################################
        # draw images
        
        x, y, z = tvec.ravel()
        draw_x, draw_y = int(x) + 300, 600 - (int(z) + 100)
        true_x, true_y = int(truePose[i][3]) + 300, 600-(int(truePose[i][11]) + 100)

        curError = np.sqrt(
            (tvec[0] - truePose[i][3]) ** 2 +
            (tvec[1] - truePose[i][7]) ** 2 +
            (tvec[2] - truePose[i][11]) ** 2)

        if (curError > maxError):
            maxError = curError
        
        if toPrint:
            print(f"Estimated position, X : {tvec[0]} , Y : {tvec[1]} , Z : {tvec[2]}")
            print(f"True position, X : {truePose[i][3]} , Y : {truePose[i][7]} , Z : {truePose[i][11]}")
            print("-")
            print("Rotation vector","[{0:02f}, {1:02f}, {2:02f}]".format(rvec[0][0], rvec[1][0], rvec[2][0]))
            print("\n-----\n")

        text = "Coordinates: x ={0:02f}m y = {1:02f}m z = {2:02f}m".format(x,y,z)
        cv2.circle(traj, (draw_x, draw_y), 1, (0, 0, 255), 2)
        cv2.circle(traj, (true_x, true_y), 1, (255, 0, 0), 2)
        cv2.rectangle(traj, (10, 30), (550, 50), (0, 0, 0), cv2.FILLED)
        cv2.putText(traj, text, (10, 50), cv2.FONT_HERSHEY_PLAIN, 1, (255, 255, 255), 1, 8)



        h1, w1 = traj.shape[:2]
        h2, w2 = curImage.shape[:2]
        vis = np.zeros((max(h1, h2), w1 + w2, 3), np.uint8)
        vis[:h1, :w1, :3] = traj

        curImage_color = np.dstack((curImage, curImage, curImage)).copy()
        for pt in track_2D.reshape(-1, 2):
            cv2.circle(curImage_color, (int(pt[0]), int(pt[1])), 3, (0, 255, 0), -1)
        vis[:h2, w1:w1 + w2, :3] = curImage_color

        cv2.imshow("Trajectory", vis)
        k = cv2.waitKey(1) & 0xFF
        if k == 27: break


    cv2.waitKey(0)
    cv2.destroyAllWindows()
    print('Maximum Error: ', maxError)
    ##### ################################ #######
    ##### END OF Print and visualize stuff #######
    ##### ########################################


In [11]:
left_img_paths = sorted(glob.glob('left_challenge/*.png'))
right_img_paths = sorted(glob.glob('right_challenge/*.png'))

K = getK()

playImageSequenceChallange(left_img_paths, right_img_paths, K)

Image:  15
Estimated position, X : [-0.47045467] , Y : [-0.13660424] , Z : [11.67728557]
True position, X : -0.7018788 , Y : -0.4239119 , Z : 12.86965
-
Rotation vector [-0.001695, 0.053389, 0.000351]

-----

Maximum Error:  [1.31130921]


Maximum Error when running the challenge code fully is 14 meters (takes around 5 minutes). For comparison running the Challange files with the old code the maximum error is 45, so this new implementation is about 3x more accurate.

Here is a screenshot from the ouput of the Challenge code. Red is the true position, and blue is our inferred position found with visual odometry.

![img](challenge_run_late.png "Title")

As you can see on the screenshot, we stay quite close to the red line. It is however clear that errors accumulate. It is very close early run, but as the run continues the errors get accumulated and thus worse. This is just the nature of this algorithm, but something you would like to improve before implementing this code into a production setup.